# B2.11 · Context engineering for the pipeline

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.10 · Severity calibration and reporting](https://spbreed.github.io/cyber-commons/lessons/B2.10.html)**.

| | |
|---|---|
| Tools used | tree-sitter, GLM-4.6, Llama 3.3, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Compare four context strategies against one bug and measure which are decidable and at what size.

**Why a security engineer needs it.** The model is given the repository and asked to be thorough, so the relevant line falls out of the window. The control it builds is: slice on the source-sink path, not on distance: the smallest context that still supports a severity decision.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Give an agent more context and it gets better, until it gets worse. The cliff is real, it arrives earlier than anyone expects, and past it you are paying more per token for a worse answer.

> **At CyberTravels.** Give the review agent CyberTravels' whole repository and it gets worse, not better. The cliff arrives earlier than anyone expects and you pay more per token for it.

## 2 · The framework

```
   accuracy
     ^
     |          .-----.
     |        .'       `.
     |      .'           `.       <- the cliff
     |    .'               `--....
     |  .'
     +--------------------------------> context tokens
        too little        enough      too much

   past the peak you pay more per token for a worse answer
```

Cross-cutting, and it applies to every stage that calls a model: stages 3, 4, 5,
7 and 14.

The instinct when a model misses something is to give it more context. Usually
the opposite is correct.

To find a vulnerability, a model needs three things: the **sink**, the
**source**, and the **path** between them. Everything else competes for
attention and for window. A repository dumped into a prompt does not produce a
thorough review — it produces a review of whatever survived truncation, and you
cannot tell which parts those were.

So context engineering is mostly subtraction, with one exception you must not
subtract: the **enclosing signature**, because that is where reachability is
decided. The identical concatenation is critical inside an HTTP handler and
irrelevant inside a migration script that takes a constant.

## 3 · The stage, as a skill

Context engineering here is not "send less" — it is finding the slice in which the defect is decidable at all, and only then making it smaller. The skill measures four candidate slices and reports which are decidable and what each carries that the defect does not depend on.

In [ ]:
# skills/appsec/context-window-sizing/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: context-window-sizing
description: >-
  Find the smallest slice of a file in which a defect is actually decidable, and
  measure what larger windows add in unrelated code. Use when tuning what a
  model is shown per finding, when analysis costs too much, or when a model
  keeps missing a bug that is on the line you gave it.
allowed-tools: Read, Grep, Glob
---

# Decidable, then small — in that order

Context engineering for an analysis pipeline is not "send less". It is finding
the slice in which the question can be answered at all, and only then making it
smaller. A ±2-line window around the bug is cheap and **not decidable**: it
lacks the signature, so nothing in it says where the value came from.

## When to use this

When designing what a pipeline sends per finding, and whenever cost per finding
is the constraint.

## Procedure

**1 — Define decidability for the defect class.** For injection: the sink, the
value's origin, and any sanitiser between them. Write it down before slicing, or
you will judge slices by how they look.

**2 — Build the candidate slices.** The whole file, a fixed window around the
line, a wider window, and a **path slice** — the enclosing function plus the
definitions it depends on. Measure each in characters.

**3 — Mark each slice decidable or not,** against step 1. Cheap and undecidable
is the trap: it looks like a saving and it produces confident answers about
information that is not there.

**4 — Count unrelated content in the decidable ones.** Functions, constants and
imports the defect does not depend on. This is what a bigger window costs, in
tokens and in the model's attention.

**5 — Pick the smallest decidable slice, and say what it excluded.** The
exclusion list is what somebody re-reads when the pipeline misses something.

## Output contract

```json
{
  "decidability": {"requires": ["str"]},
  "slices": [{"name": "str", "chars": 0, "decidable": false, "unrelated_units": 0}],
  "chosen": {"name": "str", "chars": 0, "unrelated_units": 0},
  "excluded": ["str"]
}
```

## Failure modes

- **Optimising size first.** An undecidable slice is not cheap; it is wrong at a
  lower price.
- **Judging slices by eye.** Write the decidability requirement down first.
- **Ignoring unrelated content** in a decidable slice. It is the cost you can
  actually remove.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/appsec/context-window-sizing/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/appsec/context-window-sizing/scripts/context_window_sizing.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Find the smallest slice of a file in which a defect is decidable, and measure what larger contexts add.

This is the executable half of the `context-window-sizing` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

SOURCE = '''"""Reporting service."""
import logging, os, json, datetime

log = logging.getLogger(__name__)
DEFAULT_LIMIT = 100
CACHE = {}

def _format_row(row):
    return {"id": row[0], "name": row[1], "created": str(row[2])}

def _cache_key(*parts):
    return ":".join(str(p) for p in parts)

def healthcheck():
    return {"status": "ok", "ts": datetime.datetime.utcnow().isoformat()}

def list_reports(conn, owner, limit=DEFAULT_LIMIT):
    """Called from GET /reports?owner=... — owner is user-controlled."""
    key = _cache_key("reports", owner, limit)
    if key in CACHE:
        return CACHE[key]
    rows = conn.execute("SELECT * FROM reports WHERE owner = '" + owner + "' LIMIT " + str(limit))
    out = [_format_row(r) for r in rows]
    CACHE[key] = out
    return out

def purge_cache():
    CACHE.clear()
    log.info("cache purged")
'''
lines = SOURCE.splitlines()
BUG_LINE = next(i for i, l in enumerate(lines, 1) if "SELECT * FROM reports" in l)
print(f"the bug is on line {BUG_LINE}")

def whole_file(_):     return SOURCE
def window(n, radius): return "\n".join(lines[max(n-radius-1,0):n+radius])

STRATEGIES = {"whole file": whole_file(BUG_LINE),
              "±2 line window": window(BUG_LINE, 2),
              "±6 line window": window(BUG_LINE, 6)}
for name, ctx in STRATEGIES.items():
    print(f"{name:20s}{len(ctx):>6} chars{len(ctx.splitlines()):>5} lines")

def decidable(ctx):
    """Can a reviewer judge exploitability from this context alone?"""
    return {"sink": "conn.execute" in ctx,
            "concatenation": "' + owner +" in ctx or "+ owner +" in ctx,
            "source (signature)": "def list_reports" in ctx,
            "intent (docstring)": "user-controlled" in ctx}

print(f"{'strategy':20s}{'sink':7s}{'concat':8s}{'source':8s}{'intent':8s}decidable")
print("-" * 64)
for name, ctx in STRATEGIES.items():
    d = decidable(ctx)
    ok = d["sink"] and d["concatenation"] and d["source (signature)"]
    print(f"{name:20s}{str(d['sink']):7s}{str(d['concatenation']):8s}"
          f"{str(d['source (signature)']):8s}{str(d['intent (docstring)']):8s}{ok}")
print("\nThe ±2 window has the sink and the concatenation but not the signature,")
print("so you cannot tell whether owner is user-controlled — which is the")
print("difference between critical and won't-fix.")

def path_slice(source, bug_line):
    ls = source.splitlines()
    start = max(i for i in range(bug_line) if ls[i-1].startswith("def "))
    end = next((i for i in range(start, len(ls)) if i > start and ls[i].startswith("def ")),
               len(ls))
    return "\n".join(ls[start-1:end])

sliced = path_slice(SOURCE, BUG_LINE)
print(sliced)
d = decidable(sliced)
print(f"\n{len(sliced)} chars ({len(sliced)/len(SOURCE):.0%} of the file), "
      f"decidable={d['sink'] and d['concatenation'] and d['source (signature)']}")

def evaluate(name, ctx):
    d = decidable(ctx)
    return {"strategy": name, "chars": len(ctx),
            "share": round(len(ctx)/len(SOURCE), 3),
            "decidable": d["sink"] and d["concatenation"] and d["source (signature)"],
            "noise_fns": max(ctx.count("def ") - 1, 0)}

rows = [evaluate(n, c) for n, c in STRATEGIES.items()] + [evaluate("path slice", sliced)]
print(f"{'strategy':20s}{'chars':>7}{'share':>8}{'decidable':>11}{'noise fns':>11}")
print("-" * 58)
for r in rows:
    print(f"{r['strategy']:20s}{r['chars']:>7}{r['share']:>8.0%}"
          f"{str(r['decidable']):>11}{r['noise_fns']:>11}")

best = sorted((r for r in rows if r["decidable"]), key=lambda r: r["chars"])[0]
whole = next(r for r in rows if r["strategy"] == "whole file")
print(f"\nsmallest decidable context: {best['strategy']} "
      f"({best['share']:.0%} of the file, {best['noise_fns']} unrelated functions)")
print(f"vs whole file: {1 - best['chars']/whole['chars']:.0%} smaller, "
      f"{whole['noise_fns']}→{best['noise_fns']} unrelated functions")
assert best["strategy"] == "path slice"

## What you just proved

The whole file is roughly 840 characters, the ±2 window about 200 and the path slice about 390. The ±2 window is not decidable because it lacks the signature; the ±6 window and the whole file are decidable but carry unrelated functions. The path slice is the smallest decidable context with zero unrelated functions, about 53% smaller than the whole file.

## Your turn

Apply the path-slice rule where the source is three functions away from the sink. That is the case where text windows break down entirely and the call graph from B2.1 earns its keep.

---

**Next → [B2.12 · Securing the developers' coding agents](https://spbreed.github.io/cyber-commons/lessons/B2.12.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*